# Assignment 8 - Chihuahua vs Muffin Image Classification with PyTorch

## Goal
Design your **own CNN** for binary image classification on your local PC.

You must complete **both** parts:

1. **Baseline CNN** - build, train, and evaluate a valid CNN that satisfies the minimum architecture rules.
2. **Improved CNN** - build a second CNN that is **deeper than the baseline**, train it separately, and compare the validation result.

## Very important
**Getting more than 50% validation accuracy with the BaselineCNN does NOT finish the assignment.**

The BaselineCNN is only your starting benchmark. **Part B is mandatory even if your baseline already gets 60%, 80%, 90%, or higher.**

The **50% validation accuracy is the minimum requirement for the final ImprovedCNN**, not a stopping point for the baseline.

## Important rules
- Do **not** use pretrained models such as ResNet, VGG, MobileNet, EfficientNet, etc.
- You must define the CNN architecture yourself using PyTorch layers.
- Minimum baseline architecture:
  - at least **3 `nn.Conv2d` layers**
  - at least **3 `nn.MaxPool2d` layers**
  - at least **3 `nn.Linear` layers** in the classifier
- Your ImprovedCNN must contain **more `nn.Conv2d` layers than your BaselineCNN**.
- The ImprovedCNN must still contain at least **3 MaxPool2d** and **3 Linear** layers.
- Final output must contain **2 raw logits**: Chihuahua and Muffin.
- Use `nn.CrossEntropyLoss()`; do **not** put Softmax inside the model.
- Validation images must never be used for training.

## Final submission
Submit exactly:
1. your completed `.ipynb` file
2. one PDF containing the required screenshots listed at the end of this notebook

# Step 0 - Check your local PC environment

Run this notebook using **Jupyter Notebook, JupyterLab, or VS Code** on your own PC.
Google Colab is **not required** for this version.

Before opening the notebook, install the required packages. From the assignment folder, you may run:

```text
python -m venv .venv
```

**Windows:**
```text
.venv\Scripts\activate
pip install -r Assignment_8_Local_PC_requirements.txt
jupyter notebook
```

**macOS/Linux:**
```text
source .venv/bin/activate
pip install -r Assignment_8_Local_PC_requirements.txt
jupyter notebook
```

A CUDA-capable NVIDIA GPU is optional. If PyTorch cannot use a GPU, the notebook automatically uses the CPU.

**Expected output:** Python/PyTorch information, CUDA availability, and `Device: cuda` or `Device: cpu`.

In [ ]:
import os
import sys
import platform
import random
import copy
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python version:", sys.version.split()[0])
print("Operating system:", platform.system(), platform.release())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: not used - CPU training is allowed")

# Step 1 - Download the dataset and set the local dataset path

Suggested public dataset:

**Kaggle - Muffin vs Chihuahua Image Classification**  
https://www.kaggle.com/datasets/samuelcortinhas/muffin-vs-chihuahua-image-classification

Download and extract the images on your local PC. Organize them as:

```text
Assignment8/
├── Assignment_8_Chihuahua_Muffin_Local_PC.ipynb
├── dataset/
│   ├── train/
│   │   ├── chihuahua/
│   │   └── muffins/
│   └── val/
│       ├── chihuahua/
│       └── muffins/
└── outputs/
```

The folder names inside `train/` and `val/` must match.
If your dataset is stored somewhere else, change `DATA_ROOT` in the next cell.

**Expected output:** the notebook prints the local train and validation paths and both folders are found.

In [ ]:
# Default: dataset folder is next to this notebook.
DATA_ROOT = Path('./dataset')
TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR = DATA_ROOT / 'val'
OUTPUT_DIR = Path('./outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Current working folder:', Path.cwd().resolve())
print('Train folder:', TRAIN_DIR.resolve())
print('Validation folder:', VAL_DIR.resolve())
print('Output folder:', OUTPUT_DIR.resolve())

assert TRAIN_DIR.exists(), f"Train folder not found: {TRAIN_DIR.resolve()}"
assert VAL_DIR.exists(), f"Validation folder not found: {VAL_DIR.resolve()}"

# Step 2 - Check image counts

The two classes should be reasonably balanced.

**Expected output:** image counts for `chihuahua` and `muffins` in both train and validation folders.

In [ ]:
VALID_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_images(folder):
    counts = {}
    for class_dir in sorted([p for p in folder.iterdir() if p.is_dir()]):
        counts[class_dir.name] = sum(
            1 for p in class_dir.rglob('*')
            if p.is_file() and p.suffix.lower() in VALID_EXT
        )
    return counts

train_counts = count_images(TRAIN_DIR)
val_counts = count_images(VAL_DIR)

print('Training image counts:')
for k, v in train_counts.items():
    print(f'  {k}: {v}')

print('\nValidation image counts:')
for k, v in val_counts.items():
    print(f'  {k}: {v}')

assert len(train_counts) == 2, 'Training folder must contain exactly 2 class folders.'
assert len(val_counts) == 2, 'Validation folder must contain exactly 2 class folders.'
assert set(train_counts) == set(val_counts), 'Train and validation class folder names must match.'
assert all(v > 0 for v in train_counts.values()), 'Each training class must contain images.'
assert all(v > 0 for v in val_counts.values()), 'Each validation class must contain images.'

print('\nDataset structure check: PASSED')

# Step 3 - Image transforms and DataLoader

This template uses 128 x 128 images so it can run on a normal local PC. A GPU is optional; CPU training is allowed.

Training uses simple augmentation. Validation does not use random augmentation.

**Expected output:** class names, class-to-index mapping, dataset sizes, and one batch shape such as `[32, 3, 128, 128]`.

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transform)

assert train_dataset.classes == val_dataset.classes
class_names = train_dataset.classes
num_classes = len(class_names)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

images, labels = next(iter(train_loader))

print('Classes:', class_names)
print('Class -> index:', train_dataset.class_to_idx)
print('Training images:', len(train_dataset))
print('Validation images:', len(val_dataset))
print('Batch image shape:', images.shape)
print('Batch label shape:', labels.shape)

# Step 4 - Preview training images

Run the cell to confirm that the images and labels are loaded correctly.

**Expected output:** a grid of training images with class labels.

In [ ]:
def denormalize(img):
    return (img * 0.5 + 0.5).clamp(0, 1)

images, labels = next(iter(train_loader))
n_show = min(8, len(images))

plt.figure(figsize=(12, 6))
for i in range(n_show):
    plt.subplot(2, 4, i + 1)
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    plt.imshow(img)
    plt.title(class_names[labels[i].item()])
    plt.axis('off')
plt.tight_layout()
plt.show()

# Step 5 - PART A: Design your Baseline CNN by yourself

You must complete `BaselineCNN` yourself.

## Minimum structure

```text
Input image
    ↓
Conv2d -> ReLU -> MaxPool2d
    ↓
Conv2d -> ReLU -> MaxPool2d
    ↓
Conv2d -> ReLU -> MaxPool2d
    ↓
Flatten (or Adaptive Pool + Flatten)
    ↓
Linear -> activation
    ↓
Linear -> activation
    ↓
Linear -> 2 output logits
```

You choose the channel sizes, kernel sizes, hidden units, Dropout, BatchNorm, etc.

### Rules
- at least 3 `nn.Conv2d`
- at least 3 `nn.MaxPool2d`
- at least 3 `nn.Linear`
- final output shape must be `[batch_size, 2]`
- do not copy a pretrained network
- do not put Softmax in the final layer when using CrossEntropyLoss

**Expected output after you complete the code:** the model prints successfully and the architecture checker reports `PASS`.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # ============================================================
        # TODO: DEFINE YOUR OWN BASELINE CNN HERE.
        #
        # Minimum requirements:
        #   - >= 3 nn.Conv2d layers
        #   - >= 3 nn.MaxPool2d layers
        #   - >= 3 nn.Linear layers
        #   - final output = num_classes logits
        #
        # You must choose the dimensions yourself.
        # ============================================================
        raise NotImplementedError("Student: define your BaselineCNN architecture.")

    def forward(self, x):
        # ============================================================
        # TODO: WRITE YOUR OWN FORWARD PASS HERE.
        # ============================================================
        raise NotImplementedError("Student: define the BaselineCNN forward pass.")

# After completing the class above, remove the NotImplementedError lines,
# then uncomment the next two lines.
# baseline_model = BaselineCNN(num_classes=num_classes).to(device)
# print(baseline_model)

# Step 6 - Architecture checker

Do not modify the checker.

It verifies the minimum number of PyTorch modules required by this assignment.

**Expected output:**

```text
Conv2d layers:     3 or more
MaxPool2d layers:  3 or more
Linear layers:     3 or more
Output shape:      [2, 2]
Architecture check: PASS
```

In [ ]:
def architecture_counts(model):
    return {
        'conv': sum(isinstance(m, nn.Conv2d) for m in model.modules()),
        'pool': sum(isinstance(m, nn.MaxPool2d) for m in model.modules()),
        'linear': sum(isinstance(m, nn.Linear) for m in model.modules()),
    }


def check_baseline_architecture(model):
    counts = architecture_counts(model)

    print('Conv2d layers:    ', counts['conv'])
    print('MaxPool2d layers: ', counts['pool'])
    print('Linear layers:    ', counts['linear'])

    assert counts['conv'] >= 3, 'Need at least 3 nn.Conv2d layers.'
    assert counts['pool'] >= 3, 'Need at least 3 nn.MaxPool2d layers.'
    assert counts['linear'] >= 3, 'Need at least 3 nn.Linear layers.'

    model.eval()
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
    with torch.no_grad():
        output = model(dummy)

    print('Output shape:     ', list(output.shape))
    assert output.shape == (2, num_classes), (
        f'Expected output shape [2, {num_classes}], got {list(output.shape)}'
    )

    print('Architecture check: PASS')
    return counts

# RUN THIS AFTER YOU DEFINE baseline_model:
# baseline_counts = check_baseline_architecture(baseline_model)

# Step 7 - Training and validation functions

These functions are provided. Do not mix validation images into training.

The notebook will record:
- training loss
- training accuracy
- validation loss
- validation accuracy
- best validation model

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def fit_model(model, train_loader, val_loader, epochs, learning_rate=1e-3, optimizer_name='Adam'):
    criterion = nn.CrossEntropyLoss()

    if optimizer_name.lower() == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    else:
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }

    best_val_acc = -1.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f'Epoch {epoch+1:02d}/{epochs} | '
            f'Train Loss {train_loss:.4f} | Train Acc {train_acc*100:6.2f}% | '
            f'Val Loss {val_loss:.4f} | Val Acc {val_acc*100:6.2f}%'
        )

    model.load_state_dict(best_state)
    return model, history, best_val_acc


def plot_history(history, title='Model'):
    plt.figure(figsize=(7, 4))
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title + ' - Loss')
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot([x * 100 for x in history['train_acc']], label='Train Accuracy')
    plt.plot([x * 100 for x in history['val_acc']], label='Validation Accuracy')
    plt.axhline(50, linestyle='--', label='50% Target')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title(title + ' - Accuracy')
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

print('Training functions: READY')

# Step 8 - Train the Baseline CNN

Train your BaselineCNN for **at least 5 epochs**.

You may change the learning rate or optimizer if needed.

**Expected output:** one line per epoch showing training/validation loss and accuracy, followed by the best baseline validation accuracy.

## Important
The baseline accuracy is only a **benchmark**.

- There is **no pass/fail accuracy target for the baseline**.
- If the baseline is already above 50%, you must **still complete Part B**.
- Record the best baseline validation accuracy because you will compare it with the ImprovedCNN later.

In [ ]:
BASELINE_EPOCHS = 5
BASELINE_LR = 1e-3
BASELINE_OPTIMIZER = 'Adam'

# UNCOMMENT AFTER baseline_model IS READY.
# baseline_model, baseline_history, baseline_best_val_acc = fit_model(
#     baseline_model,
#     train_loader,
#     val_loader,
#     epochs=BASELINE_EPOCHS,
#     learning_rate=BASELINE_LR,
#     optimizer_name=BASELINE_OPTIMIZER
# )
#
# print()
# print(f'Best Baseline Validation Accuracy: {baseline_best_val_acc*100:.2f}%')
#
# if baseline_best_val_acc > 0.50:
#     print('Baseline is already above 50%, but Part B is STILL REQUIRED.')
#     print('Now design, train, and evaluate a deeper ImprovedCNN.')

# Step 9 - Plot Baseline results

**Expected output:**
1. baseline training/validation loss graph
2. baseline training/validation accuracy graph

In [ ]:
# UNCOMMENT AFTER BASELINE TRAINING.
# plot_history(baseline_history, title='Baseline CNN')

# Step 10 - PART B: Design a Deeper Improved CNN

Now create a second model called `ImprovedCNN`.

## This part is mandatory
You must build and train the ImprovedCNN **even if your BaselineCNN already achieved more than 50% validation accuracy**.

## Required architecture change
Your ImprovedCNN must contain **more `nn.Conv2d` layers than your BaselineCNN**.

It must also still contain:
- at least **3 `nn.MaxPool2d` layers**
- at least **3 `nn.Linear` layers**

You may change:
- number of convolution channels
- kernel sizes
- hidden units
- activation functions
- BatchNorm
- Dropout
- learning rate
- optimizer
- number of epochs
- training augmentation

Do not use pretrained networks.

## Goal of Part B
Train a deeper model and compare it with your baseline.

- The **minimum final requirement** is **ImprovedCNN validation accuracy > 50%**.
- You should **try** to achieve validation accuracy equal to or higher than the baseline.
- A deeper network is **not guaranteed** to beat the baseline, so the improved model does not have to exceed the baseline to make the assignment valid.
- However, the ImprovedCNN must be trained, evaluated, and shown in your submission.

In [ ]:
class ImprovedCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # ============================================================
        # TODO: DEFINE YOUR OWN IMPROVED CNN HERE.
        #
        # IMPORTANT:
        #   - must have MORE Conv2d layers than your BaselineCNN
        #   - still needs >= 3 MaxPool2d and >= 3 Linear layers
        #   - final output = num_classes logits
        # ============================================================
        raise NotImplementedError("Student: define your ImprovedCNN architecture.")

    def forward(self, x):
        # ============================================================
        # TODO: WRITE YOUR OWN FORWARD PASS HERE.
        # ============================================================
        raise NotImplementedError("Student: define the ImprovedCNN forward pass.")

# After completing the class above, remove the NotImplementedError lines,
# then uncomment the next two lines.
# improved_model = ImprovedCNN(num_classes=num_classes).to(device)
# print(improved_model)

# Step 11 - Improved model architecture checker

The improved model must be deeper than the baseline in terms of convolution layers.

**Expected output:** `Improved architecture check: PASS`.

In [ ]:
def check_improved_architecture(improved_model, baseline_counts):
    counts = architecture_counts(improved_model)

    print('Baseline Conv2d layers:', baseline_counts['conv'])
    print('Improved Conv2d layers:', counts['conv'])
    print('Improved MaxPool2d layers:', counts['pool'])
    print('Improved Linear layers:', counts['linear'])

    assert counts['conv'] > baseline_counts['conv'], (
        'ImprovedCNN must contain MORE Conv2d layers than BaselineCNN.'
    )
    assert counts['pool'] >= 3, 'ImprovedCNN still needs at least 3 nn.MaxPool2d layers.'
    assert counts['linear'] >= 3, 'ImprovedCNN still needs at least 3 nn.Linear layers.'

    improved_model.eval()
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
    with torch.no_grad():
        output = improved_model(dummy)

    print('Output shape:', list(output.shape))
    assert output.shape == (2, num_classes)

    print('Improved architecture check: PASS')
    return counts

# RUN AFTER BOTH MODELS ARE READY:
# improved_counts = check_improved_architecture(improved_model, baseline_counts)

# Step 12 - Train the Improved CNN

Train the ImprovedCNN and check its validation accuracy.

You may modify the model and training settings and run the experiment again.

Examples of changes:
- add more convolution layers
- increase or decrease channel sizes
- add BatchNorm
- add or change Dropout
- change learning rate
- change Adam / SGD
- change the number of epochs
- change training augmentation

## Minimum final requirement
Your **best ImprovedCNN validation accuracy must be greater than 50%**.

If it is 50% or lower, modify the model/settings and train again.

Even if the baseline is higher than the improved result, Part B is still considered complete when:
1. the ImprovedCNN is deeper than the baseline,
2. it is trained and evaluated correctly, and
3. its best validation accuracy is greater than 50%.

In [ ]:
IMPROVED_EPOCHS = 10
IMPROVED_LR = 1e-3
IMPROVED_OPTIMIZER = 'Adam'

# UNCOMMENT AFTER improved_model IS READY.
# improved_model, improved_history, improved_best_val_acc = fit_model(
#     improved_model,
#     train_loader,
#     val_loader,
#     epochs=IMPROVED_EPOCHS,
#     learning_rate=IMPROVED_LR,
#     optimizer_name=IMPROVED_OPTIMIZER
# )
#
# print()
# print(f'Best Improved Validation Accuracy: {improved_best_val_acc*100:.2f}%')
#
# if improved_best_val_acc > 0.50:
#     print('FINAL ACCURACY REQUIREMENT: PASS (> 50%)')
# else:
#     print('FINAL ACCURACY REQUIREMENT: NOT YET')
#     print('Modify the ImprovedCNN or training settings and train again.')

# Step 13 - Compare Baseline and Improved Results

Plot the ImprovedCNN results and compare both experiments.

**Expected output:**
- ImprovedCNN loss graph
- ImprovedCNN accuracy graph
- best baseline validation accuracy
- best improved validation accuracy
- difference between the two results

## Important
The comparison is required even when the baseline already has very high accuracy.

Do not stop after the baseline. The final submission must show **both models**.

In [ ]:
# UNCOMMENT AFTER IMPROVED TRAINING.
# plot_history(improved_history, title='Improved CNN')
#
# baseline_pct = baseline_best_val_acc * 100
# improved_pct = improved_best_val_acc * 100
# difference = improved_pct - baseline_pct
#
# print(f'Baseline best validation accuracy: {baseline_pct:.2f}%')
# print(f'Improved best validation accuracy: {improved_pct:.2f}%')
# print(f'Difference: {difference:+.2f} percentage points')
#
# if improved_best_val_acc > 0.50:
#     print('FINAL ACCURACY REQUIREMENT: PASS (> 50%)')
# else:
#     print('FINAL ACCURACY REQUIREMENT: NOT YET')
#
# if improved_best_val_acc >= baseline_best_val_acc:
#     print('The ImprovedCNN matched or exceeded the baseline.')
# else:
#     print('The ImprovedCNN did not beat the baseline, but the comparison is still valid.')

# Step 14 - Save and reload the final improved model

Save the final improved model to the local `outputs/` folder and reload it.

**Expected output:** `Model reload: SUCCESS`.

In [ ]:
MODEL_PATH = OUTPUT_DIR / 'assignment8_best_model.pth'

# UNCOMMENT AFTER YOUR FINAL improved_model IS READY.
# torch.save(improved_model.state_dict(), MODEL_PATH)
# print('Saved:', MODEL_PATH.resolve())
#
# reloaded_model = ImprovedCNN(num_classes=num_classes).to(device)
# reloaded_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
# reloaded_model.eval()
# print('Model reload: SUCCESS')

# Step 15 - Validation inference examples

Use the reloaded model to classify validation images.

**Expected output:** images showing both the true label and predicted label.

In [ ]:
@torch.no_grad()
def show_validation_predictions(model, loader, n=8):
    model.eval()
    images, labels = next(iter(loader))
    images = images.to(device)
    labels = labels.to(device)

    logits = model(images)
    preds = logits.argmax(dim=1)

    n = min(n, len(images))
    plt.figure(figsize=(12, 6))

    for i in range(n):
        plt.subplot(2, 4, i + 1)
        img = denormalize(images[i].cpu()).permute(1, 2, 0).numpy()
        plt.imshow(img)
        true_name = class_names[labels[i].item()]
        pred_name = class_names[preds[i].item()]
        plt.title(f'True: {true_name}\nPred: {pred_name}')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# UNCOMMENT AFTER reloaded_model IS READY.
# show_validation_predictions(reloaded_model, val_loader, n=8)

# Final submission checklist

Run the assignment on your **local PC**, then submit exactly **two files** to Microsoft Teams.

## 1. Completed code
`StudentID_Assignment8.ipynb`

The notebook must include your own completed `BaselineCNN` and `ImprovedCNN` classes.

## 2. Screenshot PDF
`StudentID_Assignment8.pdf`

Put the following screenshots in this order:

1. Local PC environment output showing Python/PyTorch and Device
2. Dataset image counts and `Dataset structure check: PASSED`
3. Training image preview
4. Complete `BaselineCNN` code
5. Baseline model print + `Architecture check: PASS`
6. Baseline training output + best baseline validation accuracy
7. Baseline loss and accuracy graphs
8. Complete `ImprovedCNN` code
9. Improved model print + `Improved architecture check: PASS`
10. Improved training output
11. Improved loss and accuracy graphs
12. Comparison showing both baseline and improved best validation accuracy
13. Final result showing **ImprovedCNN validation accuracy > 50%**
14. `Model reload: SUCCESS` + validation prediction images

## Important
- **Part B is mandatory even when the baseline is already above 50%.**
- The 50% requirement applies to the **final ImprovedCNN**.
- You should try to match or improve the baseline, but the ImprovedCNN is not required to beat a very strong baseline.
- No written discussion is required.
- Do not submit the image dataset unless the instructor asks for it.
- Do not use a pretrained model.
- Validation images must not be used for training.